# Stepageddon - Train Step Chart Model

Train a neural network to generate DDR step charts from audio.

**Prerequisites:**
1. Run `prepare_data.py` locally to preprocess your charts
2. Zip the training data: `cd backend/ml && zip -r training_data.zip training_data/`
3. Upload `training_data.zip` directly to Colab when prompted (cell below)

**Runtime:** Select GPU (T4) under Runtime > Change runtime type

In [ ]:
# Mount Google Drive
from google.colab import drive
drive.mount('/content/drive')

In [ ]:
# Verify GPU
import torch
print(f'CUDA available: {torch.cuda.is_available()}')
if torch.cuda.is_available():
    print(f'GPU: {torch.cuda.get_device_name(0)}')
    print(f'VRAM: {torch.cuda.get_device_properties(0).total_mem / 1e9:.1f} GB')

In [ ]:
# Install dependencies
!pip install -q librosa numpy torch

In [ ]:
# Upload the ml/ module code
from google.colab import files
import os

os.makedirs('/content/ml', exist_ok=True)
# Make /content/ml a real Python package so `import ml.model` works
open('/content/ml/__init__.py', 'w').close()

print('Upload: model.py AND dataset.py (from backend/ml/)')
uploaded = files.upload()
for name, data in uploaded.items():
    out_path = os.path.join('/content/ml', os.path.basename(name))
    with open(out_path, 'wb') as f:
        f.write(data)
    print(f'Wrote {out_path}')

!ls -la /content/ml/


In [ ]:
# Upload and extract training data
# Pick ONE option below and comment out the others

# --- Option A: Copy from Google Drive to local disk (most reliable for large files) ---
# Upload training_data.zip to your Google Drive first (any location), then:
import zipfile, os
ZIP_PATH = '/content/drive/MyDrive/stepageddon/training_data.zip'  # adjust path if needed
print(f'Copying and extracting from Drive...')
with zipfile.ZipFile(ZIP_PATH, 'r') as z:
    z.extractall('/content/')
print('Done!')

# --- Option B: Direct URL download (if hosted somewhere) ---
# !wget -q -O /content/training_data.zip "YOUR_URL_HERE"
# import zipfile
# with zipfile.ZipFile('/content/training_data.zip', 'r') as z:
#     z.extractall('/content/')
# !rm /content/training_data.zip

# --- Option C: Browser upload (only works for small files <1GB) ---
# from google.colab import files
# uploaded = files.upload()
# import zipfile
# with zipfile.ZipFile(list(uploaded.keys())[0], 'r') as z:
#     z.extractall('/content/')

!ls /content/training_data/ | head -20

In [ ]:
# Verify data is accessible
import json
from pathlib import Path

DATA_DIR = Path('/content/training_data')
CHECKPOINT_DIR = Path('/content/drive/MyDrive/stepageddon/checkpoints')
CHECKPOINT_DIR.mkdir(parents=True, exist_ok=True)

manifest_path = DATA_DIR / 'manifest.json'
with open(manifest_path) as f:
    manifest = json.load(f)

print(f'Total training examples: {len(manifest)}')

# Show difficulty distribution
from collections import Counter
diff_counts = Counter(e['difficulty'] for e in manifest)
print(f'Difficulty distribution: {dict(diff_counts)}')

# Check a sample file
import numpy as np
sample = np.load(DATA_DIR / manifest[0]['filename'])
print(f"\nSample: {manifest[0]['song_title']} ({manifest[0]['difficulty']})")
print(f'  Mel shape: {sample["mel"].shape}')
print(f'  Labels shape: {sample["labels"].shape}')
print(f'  Note frames: {(sample["labels"] > 0).any(axis=1).sum()}')

In [ ]:
# Add to Python path
import sys
sys.path.insert(0, '/content')

# Also need the modules for schema imports during inference (not needed for training)
# For training only, we just need ml.model and ml.dataset

In [ ]:
# Train the model
from ml.model import StepChartModel, FocalLoss
from ml.dataset import (
    StepChartDataset,
    compute_class_weights,
    compute_default_density_per_difficulty,
    make_balanced_difficulty_sampler,
    DIFFICULTY_ID_TO_NAME,
    DENSITY_MEAN,
    DENSITY_STD,
)

import time
import random
import torch
import torch.nn as nn
from torch.utils.data import DataLoader, Subset
from torch.amp import autocast, GradScaler

# Hyperparameters
EPOCHS = 80
BATCH_SIZE = 32
LR = 3e-4
HIDDEN_DIM = 256
N_HEADS = 8
N_LAYERS = 4
CHUNK_FRAMES = 500  # 5 seconds at 100fps
WARMUP_EPOCHS = 5
VAL_SPLIT = 0.1

device = torch.device('cuda')

# Train dataset (random chunking, augmentation on)
full_dataset = StepChartDataset(
    data_dir=str(DATA_DIR),
    manifest_path=str(manifest_path),
    chunk_frames=CHUNK_FRAMES,
    is_train=True,
)

# Deterministic song-level split
rng = random.Random(42)
all_idx = list(range(len(full_dataset.entries)))
rng.shuffle(all_idx)
n_val = int(len(all_idx) * VAL_SPLIT)
val_idx = set(all_idx[:n_val])
train_idx = [i for i in all_idx if i not in val_idx]

train_dataset = Subset(full_dataset, train_idx)

# Separate val dataset with is_train=False so it uses fixed,
# non-overlapping chunks across all val songs (deterministic).
val_base = StepChartDataset(
    data_dir=str(DATA_DIR),
    manifest_path=str(manifest_path),
    chunk_frames=CHUNK_FRAMES,
    is_train=False,
)
val_base._val_chunks = [
    (i, s) for (i, s) in val_base._val_chunks if i in val_idx
]
val_dataset = val_base

n_train = len(train_dataset)
n_val_chunks = len(val_dataset)

# Per-difficulty balanced sampling: each minibatch sees all 5 difficulties
# roughly equally regardless of how skewed the manifest is.
train_sampler = make_balanced_difficulty_sampler(train_dataset, full_dataset)

train_loader = DataLoader(
    train_dataset, batch_size=BATCH_SIZE, sampler=train_sampler,
    num_workers=2, pin_memory=True, drop_last=True,
)
val_loader = DataLoader(
    val_dataset, batch_size=BATCH_SIZE, shuffle=False,
    num_workers=2, pin_memory=True,
)

print(f'Train songs: {n_train}, Val songs: {len(val_idx)}, Val chunks: {n_val_chunks}')

# Empirical mean steps/sec per difficulty — saved into the checkpoint so the
# inference path can use a data-driven default density per difficulty when
# the caller doesn't supply one explicitly.
default_density_by_id = compute_default_density_per_difficulty(
    str(manifest_path), str(DATA_DIR)
)
print(f'Empirical default density per difficulty: {default_density_by_id.tolist()}')

# Class weights
class_weights = compute_class_weights(str(manifest_path), str(DATA_DIR)).to(device)

# Model
model = StepChartModel(
    hidden_dim=HIDDEN_DIM, n_heads=N_HEADS,
    n_transformer_layers=N_LAYERS,
).to(device)
print(f'Parameters: {sum(p.numel() for p in model.parameters()):,}')

# Loss + optimizer
criterion = FocalLoss(alpha=class_weights, gamma=2.0)
optimizer = torch.optim.AdamW(model.parameters(), lr=LR, weight_decay=0.01)

import numpy as np
def lr_lambda(epoch):
    if epoch < WARMUP_EPOCHS:
        return (epoch + 1) / WARMUP_EPOCHS
    progress = (epoch - WARMUP_EPOCHS) / max(EPOCHS - WARMUP_EPOCHS, 1)
    return 0.5 * (1 + np.cos(np.pi * progress))

scheduler = torch.optim.lr_scheduler.LambdaLR(optimizer, lr_lambda)
scaler = GradScaler('cuda')

In [ ]:
# Training loop
best_val_loss = float('inf')
history = {
    'train_loss': [], 'val_loss': [], 'val_tap_f1': [],
    'per_diff_tap_f1': [], 'per_diff_pred_density': [],
}

LOG_EVERY = 20  # print every N batches
N_DIFFS = 5

for epoch in range(EPOCHS):
    t0 = time.time()

    # ---- Train ----
    model.train()
    train_loss_sum = 0.0
    train_n = 0
    n_batches = len(train_loader)
    batch_t0 = time.time()
    running_loss = 0.0
    running_n = 0
    for batch_idx, (mel, diff, density, labels) in enumerate(train_loader):
        mel = mel.to(device)
        diff = diff.to(device)
        density = density.to(device)
        labels = labels.to(device)
        optimizer.zero_grad()
        with autocast('cuda'):
            logits = model(mel, diff, density)
            loss = criterion(logits, labels)
        scaler.scale(loss).backward()
        scaler.unscale_(optimizer)
        nn.utils.clip_grad_norm_(model.parameters(), 1.0)
        scaler.step(optimizer)
        scaler.update()
        bs = mel.size(0)
        train_loss_sum += loss.item() * bs
        train_n += bs
        running_loss += loss.item() * bs
        running_n += bs

        if (batch_idx + 1) % LOG_EVERY == 0 or (batch_idx + 1) == n_batches:
            elapsed = time.time() - batch_t0
            it_per_sec = (batch_idx + 1) / elapsed
            eta = (n_batches - batch_idx - 1) / it_per_sec
            avg_loss = running_loss / running_n
            print(f'  epoch {epoch+1} [{batch_idx+1:4d}/{n_batches}] '
                  f'loss={avg_loss:.4f} | {it_per_sec:.1f} it/s | eta {eta:.0f}s', flush=True)
            running_loss = 0.0
            running_n = 0

    train_loss = train_loss_sum / train_n

    # ---- Validate ----
    model.eval()
    val_loss_sum = 0.0
    val_n = 0
    # Per-difficulty, per-class confusion stats
    tp = torch.zeros(N_DIFFS, 4, device=device)
    fp = torch.zeros(N_DIFFS, 4, device=device)
    fn = torch.zeros(N_DIFFS, 4, device=device)
    # Per-difficulty mean predicted tap density
    pred_density_sum = torch.zeros(N_DIFFS, device=device)
    pred_density_n = torch.zeros(N_DIFFS, device=device)

    print(f'  validating...', flush=True)
    with torch.no_grad():
        for mel, diff, density, labels in val_loader:
            mel = mel.to(device)
            diff = diff.to(device)
            density = density.to(device)
            labels = labels.to(device)
            with autocast('cuda'):
                logits = model(mel, diff, density)
                loss = criterion(logits, labels)
            val_loss_sum += loss.item() * mel.size(0)
            val_n += mel.size(0)

            # Per-class F1 split by difficulty
            preds = logits.argmax(dim=-1)  # [B, T, 4]
            for d in range(N_DIFFS):
                mask = (diff == d)
                if not mask.any():
                    continue
                p_d = preds[mask]
                l_d = labels[mask]
                for c in range(4):
                    tp[d, c] += ((p_d == c) & (l_d == c)).sum()
                    fp[d, c] += ((p_d == c) & (l_d != c)).sum()
                    fn[d, c] += ((p_d != c) & (l_d == c)).sum()

                # Predicted tap density per chunk for this difficulty
                probs = torch.softmax(logits[mask].float(), dim=-1)
                p_tap = probs[..., 1]  # [N, T, 4]
                seconds = mel.size(1) / (22050 / 220)
                dens = p_tap.sum(dim=(1, 2)) / seconds  # [N]
                pred_density_sum[d] += dens.sum()
                pred_density_n[d] += dens.numel()

    val_loss = val_loss_sum / val_n
    precision = tp / (tp + fp + 1e-8)
    recall = tp / (tp + fn + 1e-8)
    f1 = 2 * precision * recall / (precision + recall + 1e-8)  # [N_DIFFS, 4]

    # Aggregate (macro over difficulties) for the headline tap_f1
    overall_tap_f1 = f1[:, 1].mean().item()
    overall_hold_f1 = f1[:, 2].mean().item()

    pred_density_mean = (pred_density_sum / pred_density_n.clamp_min(1)).cpu().tolist()

    scheduler.step()
    elapsed = time.time() - t0

    history['train_loss'].append(train_loss)
    history['val_loss'].append(val_loss)
    history['val_tap_f1'].append(overall_tap_f1)
    history['per_diff_tap_f1'].append([f1[d, 1].item() for d in range(N_DIFFS)])
    history['per_diff_pred_density'].append(pred_density_mean)

    print(
        f'Epoch {epoch+1:3d}/{EPOCHS} | '
        f'train={train_loss:.4f} val={val_loss:.4f} | '
        f'tap_f1={overall_tap_f1:.3f} hold_f1={overall_hold_f1:.3f} '
        f'| {elapsed:.1f}s',
        flush=True,
    )
    # Per-difficulty breakdown. pred_density is now driven by the conditioning
    # input (which during val is the empirical per-chunk density), so it should
    # track the empirical default closely if the model is honoring the signal.
    print('  per-difficulty tap_f1 / pred_density (empirical default):')
    for d in range(N_DIFFS):
        name = DIFFICULTY_ID_TO_NAME[d]
        target = float(default_density_by_id[d])
        print(
            f'    {name:9s} f1={f1[d, 1].item():.3f}  '
            f'pred_dens={pred_density_mean[d]:.2f}/s  empirical={target:.2f}/s',
            flush=True,
        )

    # Save checkpoint every 10 epochs
    if (epoch + 1) % 10 == 0:
        torch.save({
            'epoch': epoch,
            'model_state_dict': model.state_dict(),
            'optimizer_state_dict': optimizer.state_dict(),
            'scheduler_state_dict': scheduler.state_dict(),
            'scaler_state_dict': scaler.state_dict(),
            'best_val_loss': best_val_loss,
            'default_density_by_id': default_density_by_id.tolist(),
            'args': {'hidden_dim': HIDDEN_DIM, 'n_heads': N_HEADS, 'n_layers': N_LAYERS},
        }, CHECKPOINT_DIR / f'checkpoint_epoch_{epoch+1}.pt')

    # Save best
    if val_loss < best_val_loss:
        best_val_loss = val_loss
        torch.save({
            'epoch': epoch,
            'model_state_dict': model.state_dict(),
            'val_loss': val_loss,
            'val_tap_f1_per_diff': [f1[d, 1].item() for d in range(N_DIFFS)],
            'val_pred_density_per_diff': pred_density_mean,
            'default_density_by_id': default_density_by_id.tolist(),
            'args': {'hidden_dim': HIDDEN_DIM, 'n_heads': N_HEADS, 'n_layers': N_LAYERS},
        }, CHECKPOINT_DIR / 'best_model.pt')
        print(f'  -> New best model saved (val_loss={val_loss:.4f})', flush=True)

print(f'\nTraining complete! Best val_loss: {best_val_loss:.4f}')
print(f'Best model saved at: {CHECKPOINT_DIR}/best_model.pt')

## After Training

1. Download `best_model.pt` from Google Drive (`stepageddon/checkpoints/best_model.pt`)
2. Place it in `backend/ml/checkpoints/best_model.pt`
3. Set `USE_ML_GENERATION=true` in `backend/.env`
4. Restart the backend server